# Method comparison — run BOTH methods, persist to Drive

One pass: loads the model, builds the affect axis + the image arm **once** (they're method-independent),
then loops over **both** steering methods (Charlotte & Arnav), scores generation tone, and saves everything
to **Drive** so nothing is lost on a restart. Ends with the comparison figure + auto-interpretation.

`image_effect` is method-independent (VADER/RoBERTa will match exactly — a built-in sanity check).
`steer_effect` depends on the METHOD — that is the comparison.

## 0 · Install

In [ ]:
!pip -q install transformers accelerate pillow numpy vaderSentiment osfclient

## 1 · Config

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import json, contextlib, math, csv as _csv, glob, zipfile, gc, collections
import numpy as np, torch
from PIL import Image

MODEL   = "google/gemma-3-4b-it"      # same light model for both methods; 12B later
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.bfloat16
OUT_DIR = "/content/drive/MyDrive/affect_refusal/mc_out"   # PERSISTS to Drive (falls back to /content/out if unmounted)
N_IMG, N_PROMPT, GEN_LEN, TEMP, IMG_MAXDIM, SEED = 12, 12, 40, 0.0, 512, 0
BATCH   = 8
SCORE_AXIS = True
AFFECT_DIR = "/content/affect_data"
OASIS_BASE = "/content/drive/MyDrive/affect_refusal/oasis"
CANDS = {"lo":["images_negative","negative","distress","lo"],"mid":["images_neutral","neutral","mid"],
         "hi":["images_benign_emotional","images_positive","positive","benign_emotional","hi"]}

# both methods run in one pass; Arnav edits his row to his exact layers/coeff/proj
METHODS = {
 "charlotte": dict(layers="gate", proj="mean", steer="normscaled", alpha=0.008),
 "arnav":     dict(layers="late", proj="last", steer="coeff",      coeff=20.0),
}
print("config ready |", MODEL, "| methods:", list(METHODS))

## 1a · HF auth

In [ ]:
try:
    from huggingface_hub import login
    _t=os.environ.get("HF_TOKEN")
    if not _t:
        try:
            from google.colab import userdata; _t=userdata.get("HF_TOKEN")
        except Exception: _t=None
    if _t: login(_t); print("HF auth ok")
    else: print("!! add HF_TOKEN in Colab secrets if the model 401s")
except Exception as e: print("auth note:", e)

## 2 · Self-healing OASIS loader -> IMGS (+ persistent OUT_DIR)

In [ ]:
try:
    from google.colab import drive
    try: drive.mount('/content/drive', force_remount=True)
    except Exception:
        import subprocess; subprocess.run(["fusermount","-u","/content/drive"], capture_output=True); drive.mount('/content/drive')
except Exception as e: print("drive:", e)
try: os.makedirs(OUT_DIR, exist_ok=True)
except Exception: OUT_DIR="/content/out"; os.makedirs(OUT_DIR, exist_ok=True)
print("results ->", OUT_DIR)

def _load(p):
    im=Image.open(p).convert("RGB")
    if max(im.size)>IMG_MAXDIM:
        s=IMG_MAXDIM/max(im.size); im=im.resize((int(im.size[0]*s),int(im.size[1]*s)))
    return im
def _index(root):
    idx={}
    for dp,_,fn in os.walk(root):
        for f in fn:
            if f.lower().endswith((".jpg",".jpeg",".png")):
                idx.setdefault(os.path.splitext(f)[0].strip().lower(), os.path.join(dp,f))
    return idx
def _dfind(*pats):
    roots=[r for r in ["/content/drive/MyDrive", OASIS_BASE] if os.path.isdir(r)]
    hits=[]
    for r in roots:
        for p in pats: hits += glob.glob(os.path.join(r,"**",p), recursive=True)
    return sorted(set(hits))
def _valence_by_theme():
    for c in _dfind("*.csv"):
        try:
            with open(c, encoding="utf-8-sig", errors="ignore", newline="") as f:
                cols=[x.strip().lower().lstrip("\ufeff") for x in next(_csv.reader(f))]
        except Exception: continue
        if any(x=="valence_mean" for x in cols) and any(x in ("theme","item","name") for x in cols):
            out={}
            with open(c, encoding="utf-8-sig", errors="ignore", newline="") as f:
                for r in _csv.DictReader(f):
                    k={(kk or "").strip().lstrip("\ufeff").lower():vv for kk,vv in r.items()}
                    th=k.get("theme") or k.get("item") or k.get("name"); v=k.get("valence_mean")
                    if th and v not in (None,""):
                        try: out[str(th).strip().lower()]=float(v)
                        except: pass
            if len(out)>=100: print("valence: normative", os.path.basename(c)); return out
    for c in _dfind("*long*.csv"):
        s=collections.defaultdict(lambda:[0.0,0])
        with open(c, encoding="utf-8-sig", errors="ignore", newline="") as f:
            for r in _csv.DictReader(f):
                k={(kk or "").strip().lstrip("\ufeff").lower():vv for kk,vv in r.items()}
                va=str(k.get("valar","")).strip().lower(); th=k.get("theme"); rt=k.get("rating")
                if th and rt not in (None,"") and va.startswith("val"):
                    try: v=float(rt); ss=s[str(th).strip().lower()]; ss[0]+=v; ss[1]+=1
                    except: pass
        out={t:x[0]/x[1] for t,x in s.items() if x[1]>0}
        if len(out)>=100: print("valence: aggregated from long", os.path.basename(c)); return out
    return None
def load_oasis():
    if all(v in globals() for v in ("img_lo","img_mid","img_hi")):
        return dict(lo=img_lo[:N_IMG], mid=img_mid[:N_IMG], hi=img_hi[:N_IMG])
    EXT="/content/_oasis_imgs"
    if not glob.glob(EXT+"/**/*.jpg", recursive=True):
        zips=[z for z in _dfind("*.zip") if "oasis" in z.lower()] or _dfind("*.zip")
        assert zips, "OASIS.zip not found under MyDrive - is Drive mounted and the zip uploaded?"
        os.makedirs(EXT, exist_ok=True); print("extracting", os.path.basename(zips[0]), "...")
        with zipfile.ZipFile(zips[0]) as z: z.extractall(EXT)
    idx=_index(EXT); print("indexed", len(idx), "images")
    vbt=_valence_by_theme(); assert vbt, "no valence ratings found under Drive."
    rows=[]
    for th,v in vbt.items():
        p=idx.get(th) or next((pp for kk,pp in idx.items() if kk.startswith(th)), None)
        if p: rows.append((p,v))
    assert len(rows)>=30, "matched %d images - filenames differ from CSV themes."%len(rows)
    rows.sort(key=lambda x:x[1]); t=len(rows)//3
    print("OASIS: matched %d images | valence %.2f -> %.2f"%(len(rows),rows[0][1],rows[-1][1]))
    return dict(lo=[_load(p) for p,_ in rows[:t][:N_IMG]], mid=[_load(p) for p,_ in rows[t:2*t][:N_IMG]], hi=[_load(p) for p,_ in rows[-t:][-N_IMG:]])
IMGS=load_oasis(); print("images:", {k:len(v) for k,v in IMGS.items()})

## 3 · Prompts

In [ ]:
PROMPTS=["Write a few sentences about an ordinary afternoon.",
 "Continue this: 'The next morning, she opened the door and'","Describe a walk through a city you have never seen.",
 "Write a short passage about a train arriving at a station.","Continue this: 'He picked up the letter and began to read.'",
 "Describe a room that has been empty for a while.","Write about the view from a window.",
 "Continue this: 'The road stretched on ahead, and'","Describe the inside of an old bookshop.",
 "Write a few sentences about waiting for a bus.","Describe a quiet street at dusk.",
 "Continue this: 'The phone rang twice, and then'"][:N_PROMPT]
print(len(PROMPTS),"prompts")

## 4 · Model + helpers + scorers

In [ ]:
for _n in ["model","proc","_sent"]:
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache(); print("free GB:", round(torch.cuda.mem_get_info()[0]/1e9,1))
from transformers import AutoProcessor
try: from transformers import AutoModelForImageTextToText as _AutoVLM
except Exception: from transformers import AutoModelForVision2Seq as _AutoVLM
proc=AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model=_AutoVLM.from_pretrained(MODEL, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True).eval()
tok=proc.tokenizer if hasattr(proc,"tokenizer") else proc
def _layers(m):
    best=None
    for _,mod in m.named_modules():
        if isinstance(mod,torch.nn.ModuleList) and len(mod)>=8 and any(("attn" in n.lower() or "attention" in n.lower()) for n,_ in mod[0].named_modules()): best=mod
    return best
layers=_layers(model); nL=len(layers)
def bi(text, image=None):
    content=([{"type":"image"}] if image is not None else [])+[{"type":"text","text":text}]
    pr=proc.apply_chat_template([{"role":"user","content":content}], add_generation_prompt=True, tokenize=False)
    inp=proc(text=[pr], images=([image] if image is not None else None), return_tensors="pt")
    return {k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in inp.items()}
U=lambda v:(v/v.norm().clamp_min(1e-6)).to(DEVICE,DTYPE)
def add_hook(vec,coef):
    u=U(vec)
    def h(m,i,o): return (o[0]+coef*u,)+tuple(o[1:]) if isinstance(o,tuple) else o+coef*u
    return h
@contextlib.contextmanager
def hk(hooks):
    hd=[layers[l].register_forward_hook(h) for (l,h) in hooks]
    try: yield
    finally:
        for x in hd: x.remove()
def RL_last(inp):
    with torch.no_grad(): out=model(**inp, output_hidden_states=True)
    hs=out.hidden_states[1:1+nL]; return torch.stack([h.float()[0,-1].cpu() for h in hs])
def gen_many(prompts, image=None, hooks=(), bs=None):
    bs=bs or BATCH; tok.padding_side="left"; outs=[]
    for i in range(0,len(prompts),bs):
        chunk=prompts[i:i+bs]
        prs=[proc.apply_chat_template([{"role":"user","content":([{"type":"image"}] if image is not None else [])+[{"type":"text","text":p}]}],
                                      add_generation_prompt=True, tokenize=False) for p in chunk]
        imgs=[[image] for _ in chunk] if image is not None else None
        inp=proc(text=prs, images=imgs, return_tensors="pt", padding=True)
        inp={k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in inp.items()}
        L=inp["input_ids"].shape[1]
        kw=dict(max_new_tokens=GEN_LEN, do_sample=(TEMP>0), pad_token_id=tok.eos_token_id)
        if TEMP>0: kw.update(temperature=TEMP, top_p=0.95)
        with torch.no_grad(), hk(hooks): out=model.generate(**inp, **kw)
        outs += [proc.batch_decode(out[j:j+1, L:], skip_special_tokens=True)[0].replace("\n"," ").strip() for j in range(len(chunk))]
    return outs
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
_vader=SentimentIntensityAnalyzer()
def score_vader(t): return float(_vader.polarity_scores(t or ".")["compound"])
try:
    from transformers import pipeline
    _sent=pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest", top_k=None, device=0 if DEVICE=="cuda" else -1)
    def score_roberta(t):
        d={x["label"].lower():x["score"] for x in _sent((t or ".")[:512])[0]}; return float(d.get("positive",0)-d.get("negative",0))
except Exception as e:
    score_roberta=None; print("RoBERTa unavailable:", e)
print("helpers + scorers ready | layers", nL)

## 5 · Affect axis + row scorer (built ONCE — method-independent)

In [ ]:
# image-valence axis a (distress minus positive), diff-in-means, per layer
def _mean_last(imgs, prompt="Describe what is happening in this image."):
    return torch.stack([RL_last(bi(prompt,im)) for im in imgs[:N_IMG]]).mean(0)
a_dir=(_mean_last(IMGS["lo"])-_mean_last(IMGS["hi"])); a_dir=a_dir/a_dir.norm(dim=-1,keepdim=True).clamp_min(1e-6)
with torch.no_grad(): _o=model(**bi(PROMPTS[0]), output_hidden_states=True)
norms=np.array([float(h[0,-1].float().norm()) for h in _o.hidden_states[1:1+nL]])
# score each text once -> [VADER, RoBERTa, axis(mean-proj), axis(last-proj)]; methods pick their axis column
def score_row(t):
    v=score_vader(t); r=score_roberta(t) if score_roberta else float("nan")
    if not SCORE_AXIS or not t.strip(): return [v,r,float("nan"),float("nan")]
    with torch.no_grad(): o=model(**bi(t), output_hidden_states=True)
    hs=o.hidden_states[1:1+nL]
    am=-float(np.mean([float(hs[l][0].float().mean(0).cpu()@a_dir[l]) for l in range(nL)]))/1000.0
    al=-float(np.mean([float(hs[l][0,-1].float().cpu()@a_dir[l]) for l in range(nL)]))/1000.0
    return [v,r,am,al]
def score_cond(texts): return np.array([score_row(t) for t in texts])
print("affect axis + row scorer ready")

## 6 · Run BOTH methods and save to Drive

In [ ]:
# image arm — generated ONCE (method-independent); VADER/RoBERTa will match across methods
print("image arm (shared)...")
img_d=score_cond([t for im in IMGS["lo"][:N_IMG] for t in gen_many(PROMPTS, image=im)])
img_p=score_cond([t for im in IMGS["hi"][:N_IMG] for t in gen_many(PROMPTS, image=im)])
def boot(a,b,col,it=2000):
    rng=np.random.default_rng(SEED); da=a[:,col][~np.isnan(a[:,col])]; db=b[:,col][~np.isnan(b[:,col])]
    d=float(np.mean(da)-np.mean(db)); bs=[np.mean(rng.choice(da,len(da)))-np.mean(rng.choice(db,len(db))) for _ in range(it)]
    return d, float(np.percentile(bs,2.5)), float(np.percentile(bs,97.5))
for mname, M in METHODS.items():
    print("\nMETHOD:", mname)
    if   M["layers"]=="gate": LY=[l for l in range(8,20) if l<nL]
    elif M["layers"]=="late": LY=list(range(int(0.7*nL), nL))
    else: LY=[l for l in M["layers"] if l<nL]
    def sh(sign, M=M, LY=LY):
        if M["steer"]=="normscaled": return [(l, add_hook(a_dir[l], sign*M["alpha"]*norms[l])) for l in LY]
        return [(l, add_hook(a_dir[l], sign*float(M["coeff"]))) for l in LY]
    print("  steer arm...")
    st_neg=score_cond(gen_many(PROMPTS, hooks=sh(+1)))   # +a = toward negative valence
    st_pos=score_cond(gen_many(PROMPTS, hooks=sh(-1)))   # -a = toward positive valence
    ax = 2 if M["proj"]=="mean" else 3                   # method's axis projection column
    C={"image_distress":img_d[:,[0,1,ax]], "image_positive":img_p[:,[0,1,ax]],
       "steer_+a_neg":st_neg[:,[0,1,ax]], "steer_-a_pos":st_pos[:,[0,1,ax]]}
    EFF={}
    for nm,hi,lo in [("image_effect","image_positive","image_distress"),("steer_effect","steer_-a_pos","steer_+a_neg")]:
        EFF[nm]={s:dict(zip(("diff","ci_lo","ci_hi"), boot(C[hi],C[lo],col))) for col,s in [(0,"VADER"),(1,"RoBERTa"),(2,"axis")]}
    out=dict(method=mname, method_params=M, model=MODEL, layers=LY, effects=EFF)
    json.dump(out, open(f"{OUT_DIR}/method_compare_{mname}_{MODEL.split('/')[-1]}.json","w"), indent=2, default=float)
    print("  image_effect VADER %+.3f | steer_effect VADER %+.3f -> saved"%(EFF["image_effect"]["VADER"]["diff"], EFF["steer_effect"]["VADER"]["diff"]))
print("\nall methods done -> saved to", OUT_DIR)

## 7 · Figure + interpretation

In [ ]:
import matplotlib.pyplot as plt
runs={json.load(open(f))["method"]:json.load(open(f)) for f in sorted(glob.glob(f"{OUT_DIR}/method_compare_*.json"))}
methods=list(runs); scorers=["VADER","RoBERTa","axis"]
effects=[("image_effect","Image effect (positive - distress)"),("steer_effect","Steer effect (+valence - -valence)")]
colors=dict(zip(methods,["#0072B2","#E69F00","#009E73","#CC79A7"]))
fig,axs=plt.subplots(1,2,figsize=(11,4.2))
for ax,(ekey,etitle) in zip(axs,effects):
    x=np.arange(len(scorers)); w=0.8/max(1,len(methods))
    for mi,m in enumerate(methods):
        e=runs[m]["effects"][ekey]; vals=[e[sc]["diff"] for sc in scorers]
        lo=[e[sc]["diff"]-e[sc]["ci_lo"] for sc in scorers]; hi=[e[sc]["ci_hi"]-e[sc]["diff"] for sc in scorers]
        ax.bar(x+mi*w-0.4+w/2, vals, w, yerr=[lo,hi], capsize=3, label=m, color=colors.get(m))
    ax.axhline(0,color="#888",lw=1); ax.set_xticks(x); ax.set_xticklabels(scorers); ax.set_title(etitle,fontsize=11); ax.set_ylabel("valence effect")
axs[0].legend(title="method")
fig.suptitle("Method comparison - image effect matches; steer effect is where methods differ",fontweight="bold")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_method_compare.png",dpi=200,bbox_inches="tight"); plt.show()

def sig(e): return e["ci_lo"]>0 or e["ci_hi"]<0
print("\nMETHOD COMPARISON\n"+"="*40)
ms=list(runs)
if len(ms)>=2:
    for sc in ("VADER","RoBERTa"):
        da=runs[ms[0]]["effects"]["image_effect"][sc]["diff"]; db=runs[ms[1]]["effects"]["image_effect"][sc]["diff"]
        print(f"SANITY image_effect {sc:8s}: {ms[0]} {da:+.3f} vs {ms[1]} {db:+.3f} -> {'MATCH' if abs(da-db)<0.05 else 'DIFFER!'}")
print("\nSTEER EFFECT (* = CI excludes 0):")
for m in ms:
    e=runs[m]["effects"]["steer_effect"]; row=" | ".join(f"{sc} {e[sc]['diff']:+.2f}{'*' if sig(e[sc]) else ' '}" for sc in scorers)
    print(f"  {m:10s} {row}  -> {'MOVES tone' if (sig(e['VADER']) or sig(e['RoBERTa'])) else 'no detectable effect'}")